# MPD CUDA BPR-MF 학습

Google Colab에서 `bpr_mf_cuda.py`를 실행하기 위한 노트북입니다. 시작하기 전에 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하세요.

In [ ]:
# Colab에 설치된 PyTorch가 GPU를 정상적으로 인식하는지 확인한다.
import torch

print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
assert torch.cuda.is_available(), "런타임 유형을 GPU로 변경한 뒤 다시 실행하세요."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# 프로젝트, MPD 데이터, 학습 결과를 사용할 Google Drive를 연결한다.
from google.colab import drive

drive.mount("/content/drive")

## 경로 설정

배포 폴더와 MPD 데이터를 Google Drive에 올린 뒤, 실제 폴더 위치에 맞게 아래 경로를 수정하세요.

In [ ]:
# 본인의 Google Drive 폴더 구조에 맞게 세 경로를 수정한다.
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/bpr_mf_cuda_release")
DATA_DIR = Path("/content/drive/MyDrive/mpd/data")
OUTPUT_DIR = Path("/content/drive/MyDrive/bpr_mf_outputs")

SCRIPT_PATH = PROJECT_DIR / "src/bpr_mf_cuda.py"
assert SCRIPT_PATH.is_file(), f"실행 파일을 찾을 수 없습니다: {SCRIPT_PATH}"
assert DATA_DIR.is_dir(), f"데이터 폴더를 찾을 수 없습니다: {DATA_DIR}"
slice_count = len(list(DATA_DIR.glob("mpd.slice.*.json")))
assert slice_count > 0, "MPD JSON slice 파일을 찾을 수 없습니다."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("실행 파일:", SCRIPT_PATH)
print("MPD slice 수:", slice_count)
print("결과 저장 폴더:", OUTPUT_DIR)

## 학습 설정

처음에는 `MAX_SLICES = 1`, `EPOCHS = 1`로 실행을 확인하고 값을 늘리는 것을 권장합니다. `MAX_SLICES = 0`이면 전체 MPD를 사용합니다.

In [ ]:
# 실험 규모와 BPR-MF 하이퍼파라미터를 설정한다.
MAX_SLICES = 10
EPOCHS = 10
BATCH_SIZE = 8192
FACTORS = 64
LEARNING_RATE = 0.01
REGULARIZATION = 0.0025
EVAL_NEGATIVES = 100
SEED = 42

# True이면 interaction을 CPU에 보관해 GPU 메모리 사용량을 줄인다.
CPU_INTERACTIONS = True

# None이면 epoch마다 모든 학습 interaction 수만큼 샘플링한다.
SAMPLES_PER_EPOCH = None

OUTPUT_PATH = OUTPUT_DIR / f"bpr_mf_slices_{MAX_SLICES}.pt"
print("체크포인트 저장 경로:", OUTPUT_PATH)

## 학습 및 평가

학습이 끝나면 validation/test의 HitRate@K, NDCG@K, sampled AUC가 출력되고 체크포인트가 Google Drive에 저장됩니다.

In [ ]:
# 설정값으로 실행 명령을 구성하고 학습 프로세스를 시작한다.
import subprocess
import sys

command = [
    sys.executable,
    str(SCRIPT_PATH),
    "--data-dir", str(DATA_DIR),
    "--output", str(OUTPUT_PATH),
    "--max-slices", str(MAX_SLICES),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--factors", str(FACTORS),
    "--lr", str(LEARNING_RATE),
    "--reg", str(REGULARIZATION),
    "--eval-negatives", str(EVAL_NEGATIVES),
    "--seed", str(SEED),
]

if CPU_INTERACTIONS:
    command.append("--cpu-interactions")
if SAMPLES_PER_EPOCH is not None:
    command.extend(["--samples-per-epoch", str(SAMPLES_PER_EPOCH)])

print("실행 명령:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
# 저장된 체크포인트의 기본 정보와 평가 결과를 확인한다.
checkpoint = torch.load(OUTPUT_PATH, map_location="cpu", weights_only=False)

print("저장 파일:", OUTPUT_PATH)
print("플레이리스트 수:", len(checkpoint["playlist_ids"]))
print("트랙 수:", len(checkpoint["track_uris"]))
print("Validation:", checkpoint["validation_metrics"])
print("Test:", checkpoint["test_metrics"])